# فصل ۸: تحلیل قابلیت فرآیندهای غیرنرمال در تولید گندله سنگ آهن
## بررسی مقاومت فشاری سرد (CCS) و مقایسه رویکردهای آماری در پایتون
مؤلف: امین — کنترل کیفیت و تحلیل داده‌های صنعتی

دیتاست: [`Chapter08_Pellet_Nonnormal_Capability_Dataset.xlsx`](../data/Chapter08_Pellet_Nonnormal_Capability_Dataset.xlsx)

### ۰. بارگذاری کتابخانه‌ها


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['axes.unicode_minus'] = False

### ۱. بارگذاری ۱۰۰ داده تجربی CCS فصل هشتم


In [ ]:
excel_path = '../data/Chapter08_Pellet_Nonnormal_Capability_Dataset.xlsx'
df_100 = pd.read_excel(excel_path, sheet_name='chapter8_ccs_100')
ccs = df_100['CCS_kg'].dropna().values
LSL = 250.0

print(f'تعداد نمونه: {len(ccs)}')
print(f'میانگین: {np.mean(ccs):.2f} kg/pellet')
print(f'انحراف معیار: {np.std(ccs, ddof=1):.2f}')
print(f'چولگی: {stats.skew(ccs, bias=False):.3f}')
print(f'کشیدگی اضافی: {stats.kurtosis(ccs, fisher=True, bias=False):.3f}')

### ۲. آزمون اندرسون-دارلینگ (Anderson-Darling Test)


In [ ]:
ad_res = stats.anderson(ccs, dist='norm')
print(f'آماره Anderson-Darling: {ad_res.statistic:.4f}')
for sl, cv in zip(ad_res.significance_level, ad_res.critical_values):
    print(f'سطح اطمینان {sl}%: مقدار بحرانی = {cv:.3f}')

### ۳. تبدیل باکس-کاکس (Box-Cox Transformation)


In [ ]:
y, lam = stats.boxcox(ccs)
print(f'بهترین لاندا برآورد شده: {lam:.4f}')

mean_y = np.mean(y)
std_y = np.std(y, ddof=1)
lsl_y = (LSL**lam - 1) / lam
cpl_bc = (mean_y - lsl_y) / (3 * std_y)
defect_bc = stats.norm.cdf(lsl_y, loc=mean_y, scale=std_y)

print(f'شاخص Cpl باکس-کاکس: {cpl_bc:.4f}')
print(f'نرخ عدم انطباق تخمینی با باکس-کاکس: {defect_bc * 100:.2f}%')

### ۴. برازش ویبول و شاخص Cpl معادل


In [ ]:
shape, loc, scale = stats.weibull_min.fit(ccs, floc=0)
p_below = stats.weibull_min.cdf(LSL, shape, loc=loc, scale=scale)
cpl_weib = stats.norm.ppf(1 - p_below) / 3
print(f'پارامتر شکل ویبول: {shape:.3f} | مقیاس: {scale:.2f}')
print(f'نرخ عدم انطباق ویبول: {p_below*100:.2f}%')
print(f'Cpl معادل ویبول: {cpl_weib:.4f}')

### ۵. رسم هیستوگرام و مقایسه برازش قبل و بعد از تبدیل


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# داده‌های اصلی
sns.histplot(ccs, kde=True, ax=axes[0], color='royalblue', bins=15)
axes[0].axvline(LSL, color='red', linestyle='--', label=f'LSL = {LSL}')
axes[0].set_title('Raw CCS Distribution (Skewed)')
axes[0].legend()

# داده‌های تبدیل‌شده باکس-کاکس
sns.histplot(y, kde=True, ax=axes[1], color='darkgreen', bins=15)
axes[1].axvline(lsl_y, color='red', linestyle='--', label=f'Transformed LSL = {lsl_y:.4f}')
axes[1].set_title(f'Box-Cox Transformed (lambda = {lam:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()